# Notebook 59 — Cluster Optimization

**Cluster optimization specifies global adaptive serving.**

Notebook 59 closes the second systems arc. Notebooks 43, 49, and 53 build the path from controller policy to infrastructure to distributed scheduling. This notebook turns those pieces into a cluster-level optimization loop: local adaptive serving becomes global adaptive serving through bounded policy updates.

In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import ListedColormap, BoundaryNorm

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 180,
    "font.size": 13,
    "axes.titlesize": 28,
    "axes.labelsize": 18,
    "legend.fontsize": 12,
})

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, w, h, text, fontsize=14, lw=1.8):
    x, y = xy
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.08",
        linewidth=lw,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(patch)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize)
    return patch

def arrow(ax, start, end, connectionstyle="arc3,rad=0.0", lw=1.8):
    arr = FancyArrowPatch(
        start, end,
        arrowstyle="->",
        mutation_scale=16,
        linewidth=lw,
        color="black",
        connectionstyle=connectionstyle
    )
    ax.add_patch(arr)
    return arr

## 1. Architecture: cluster optimization specifies global adaptive serving

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Cluster optimization specifies global adaptive serving", pad=22)

rounded_box(ax, (0.36, 0.79), 0.28, 0.09, "Cluster\nObjective U(S)", fontsize=17)
rounded_box(ax, (0.34, 0.63), 0.32, 0.10, "Global\nOptimizer π*(S)", fontsize=17)

for x, label in [(0.10, "Region A\nScheduler"), (0.39, "Region B\nScheduler"), (0.68, "Region C\nScheduler")]:
    rounded_box(ax, (x, 0.42), 0.22, 0.10, label, fontsize=15)

for x, label in [(0.10, "GPU\nworkers"), (0.39, "GPU\nworkers"), (0.68, "GPU\nworkers")]:
    rounded_box(ax, (x, 0.23), 0.22, 0.10, label, fontsize=15)

rounded_box(ax, (0.21, 0.07), 0.58, 0.08, "Telemetry Fabric", fontsize=16)

arrow(ax, (0.50, 0.79), (0.50, 0.73))
arrow(ax, (0.50, 0.63), (0.21, 0.52), "arc3,rad=0.10")
arrow(ax, (0.50, 0.63), (0.50, 0.52))
arrow(ax, (0.50, 0.63), (0.79, 0.52), "arc3,rad=-0.10")

for x in [0.21, 0.50, 0.79]:
    arrow(ax, (x, 0.42), (x, 0.33))
    arrow(ax, (x, 0.23), (0.50, 0.15), "arc3,rad=0.12" if x < 0.5 else ("arc3,rad=-0.12" if x > 0.5 else "arc3,rad=0"))

arrow(ax, (0.50, 0.15), (0.50, 0.63), "arc3,rad=-0.28")
ax.text(
    0.5, 0.005,
    "Local adaptation becomes cluster-wide optimization through bounded policy updates.",
    ha="center", va="bottom", fontsize=16
)

savefig("59_architecture_v3.png")

## 2. State evolution: telemetry becomes policy

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Cluster state updates the next policy", pad=22)

steps = [
    ("Telemetry", 0.80),
    ("Cluster state\nS(t)", 0.66),
    ("Objective\nU(S)", 0.52),
    ("Policy\nπ(t)", 0.38),
    ("Distributed\nschedulers", 0.24),
    ("Updated\ntelemetry", 0.10),
]
for label, y in steps:
    rounded_box(ax, (0.34, y), 0.32, 0.08, label, fontsize=16)

for (_, y1), (_, y2) in zip(steps[:-1], steps[1:]):
    arrow(ax, (0.50, y1), (0.50, y2 + 0.08))

ax.text(0.73, 0.50, r"$S(t+1)=F(S(t),\pi(t))$", fontsize=22, va="center")
arrow(ax, (0.34, 0.14), (0.34, 0.80), "arc3,rad=0.38")

ax.text(
    0.50, 0.01,
    "The cluster is a controlled dynamical system: state selects policy; policy changes state.",
    ha="center", fontsize=15
)

savefig("59_state_evolution.png")

## 3. Optimization objective: balance benefit against latency, memory, spillover, and verification cost

In [ ]:
x = np.linspace(0, 1, 400)
throughput = 1.22 * (1 - np.exp(-3.2 * x))
latency = 0.20 + 1.04 * x**3.0
memory = 0.11 + 0.18 * np.maximum(0, x - 0.62)**2.2 / (0.38**2.2)
spillover = 0.14 + 0.23 * np.maximum(0, x - 0.66)**2.0 / (0.34**2.0)
verification = 0.04 + 0.08 * x
objective = throughput - 0.45*latency - 0.45*memory - 0.40*spillover - 0.25*verification
best = x[np.argmax(objective)]

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(x, throughput, label="throughput benefit")
ax.plot(x, latency, label="latency pressure")
ax.plot(x, spillover, label="failure spillover risk")
ax.plot(x, objective, linewidth=2.5, label="cluster objective")
ax.axvline(best, linestyle="--", color="black", label=f"best ≈ {best:.2f}")
ax.set_title("Cluster objective balances gain, latency, and spillover")
ax.set_xlabel("global optimization intensity")
ax.set_ylabel("normalized value")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")
ax.text(
    0.50, -0.18,
    r"$\pi^*=\arg\max_\pi \left[G(\pi)-L(\pi)-M(\pi)-S(\pi)-V(\pi)\right]$",
    transform=ax.transAxes, ha="center", fontsize=18
)

savefig("59_cluster_objective.png")

## 4. Convergence: node utilization approaches the cluster target

In [ ]:
steps = np.arange(0, 31, 3)
target = 0.72

initials = np.array([0.92, 0.51, 0.81, 0.62, 0.38, 0.76])
rates = np.array([0.30, 0.25, 0.24, 0.20, 0.29, 0.22])

fig, ax = plt.subplots(figsize=(12, 7))
for i, (init, rate) in enumerate(zip(initials, rates), start=1):
    y = target + (init - target) * np.exp(-rate * steps)
    ax.plot(steps, y, marker="o", label=f"node {i}")

ax.axhline(target, color="black", linestyle="--", label="cluster target")
ax.axhspan(0.82, 1.02, alpha=0.06, label="overload band")
ax.axhspan(0.64, 0.80, alpha=0.04, label="operating band")
ax.set_title("Node utilization balancing across the cluster")
ax.set_xlabel("coordination step")
ax.set_ylabel("GPU utilization")
ax.set_ylim(0.30, 1.02)
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, loc="upper right")

savefig("59_node_utilization_balancing.png")

## 5. Policy surface: selected cluster policy depends on load imbalance and reserve capacity

In [ ]:
reserve = np.linspace(0.05, 0.95, 160)
imbalance = np.linspace(0.05, 0.95, 160)
R, I = np.meshgrid(reserve, imbalance)

# Policy encoding:
# 0 steady, 1 reserve, 2 shed/shorten, 3 rebalance, 4 scale-out
policy = np.zeros_like(R, dtype=int)

policy[(R > 0.58) & (I < 0.43)] = 1
policy[(R < 0.34) & (I > 0.55)] = 2
policy[(I > 0.45) & (R >= 0.34)] = 3
policy[(I > 0.75) & (R > 0.74)] = 4
policy[(R < 0.34) & (I > 0.55)] = 2

labels = ["steady", "reserve", "shed/shorten", "rebalance", "scale-out"]
cmap = ListedColormap(["#440154", "#fde725", "#35b779", "#31688e", "#21918c"])
norm = BoundaryNorm(np.arange(-0.5, 5.5, 1), cmap.N)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.pcolormesh(R, I, policy, shading="auto", cmap=cmap, norm=norm)
cbar = plt.colorbar(im, ax=ax, ticks=np.arange(5))
cbar.ax.set_yticklabels(labels)
cbar.set_label("selected cluster policy")

ax.text(0.19, 0.26, "steady", fontsize=17)
ax.text(0.70, 0.25, "reserve", fontsize=17)
ax.text(0.20, 0.74, "shed /\nshorten", fontsize=17)
ax.text(0.52, 0.60, "rebalance", fontsize=17)
ax.text(0.79, 0.86, "scale-out", fontsize=17)

ax.set_title("Cluster optimization policy map")
ax.set_xlabel("reserve capacity")
ax.set_ylabel("load imbalance")
ax.set_xlim(0.05, 0.95)
ax.set_ylim(0.05, 0.95)

savefig("59_policy_surface_v3.png")

## 6. Complete loop: observe, evaluate, optimize, broadcast, execute

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Global optimization loop", pad=22)

nodes = {
    "Observe": (0.40, 0.78),
    "Aggregate\ncluster state": (0.67, 0.60),
    "Evaluate\nobjective": (0.61, 0.30),
    "Optimize\nplacement": (0.40, 0.12),
    "Broadcast\npolicy": (0.17, 0.30),
    "Local\nexecution": (0.10, 0.60),
}
for label, (x0, y0) in nodes.items():
    rounded_box(ax, (x0, y0), 0.24, 0.11, label, fontsize=16)

arrow(ax, (0.64, 0.83), (0.67, 0.66), "arc3,rad=-0.23")
arrow(ax, (0.79, 0.60), (0.73, 0.41))
arrow(ax, (0.61, 0.30), (0.52, 0.23), "arc3,rad=-0.18")
arrow(ax, (0.40, 0.12), (0.29, 0.30), "arc3,rad=-0.16")
arrow(ax, (0.17, 0.30), (0.20, 0.60))
arrow(ax, (0.34, 0.65), (0.46, 0.78), "arc3,rad=0.17")

ax.text(
    0.50, 0.48,
    r"$\max_{\pi}\;U(\pi)=G(\pi)-L(\pi)-M(\pi)-S(\pi)-V(\pi)$",
    ha="center", fontsize=22
)
ax.text(
    0.50, 0.02,
    "Cluster policy is an updated control loop, not a static rule.",
    ha="center", fontsize=15
)

savefig("59_global_optimization_loop.png")

## 7. Failure recovery: local fallback prevents cluster-wide failure

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Failure recovery policy", pad=22)

top = [
    ("node\nhealthy", 0.05),
    ("overload /\nfailure", 0.25),
    ("telemetry\nalert", 0.45),
    ("traffic\nreroute", 0.65),
    ("recover /\nrebalance", 0.85),
]
for label, x0 in top:
    rounded_box(ax, (x0, 0.58), 0.16, 0.12, label, fontsize=15)
for (_, x1), (_, x2) in zip(top[:-1], top[1:]):
    arrow(ax, (x1+0.16, 0.64), (x2, 0.64))

rounded_box(ax, (0.39, 0.28), 0.26, 0.11, "fallback:\nlocal shorten", fontsize=15)
rounded_box(ax, (0.66, 0.28), 0.24, 0.11, "global:\ncapacity shift", fontsize=15)
arrow(ax, (0.53, 0.58), (0.52, 0.39))
arrow(ax, (0.65, 0.335), (0.66, 0.335))
arrow(ax, (0.78, 0.39), (0.73, 0.58), "arc3,rad=-0.25")

ax.text(
    0.5, 0.08,
    "Cluster optimization keeps local failures from becoming cluster-wide failures.",
    ha="center", fontsize=16
)

savefig("59_failure_recovery_policy.png")

## 8. Replica placement: heterogeneous nodes receive different roles

In [ ]:
nodes = ["node A", "node B", "node C", "node D", "node E", "node F"]
draft = np.array([0.50, 0.32, 0.40, 0.24, 0.55, 0.28])
verify = np.array([0.22, 0.38, 0.25, 0.30, 0.16, 0.36])
standby = np.array([0.18, 0.20, 0.25, 0.36, 0.19, 0.26])
telemetry = 1 - draft - verify - standby

parts = [
    ("draft replicas", draft),
    ("verification replicas", verify),
    ("standby replicas", standby),
    ("telemetry", telemetry),
]

fig, ax = plt.subplots(figsize=(12, 7))
left = np.zeros(len(nodes))
for label, vals in parts:
    ax.barh(nodes, vals, left=left, label=label)
    for i, v in enumerate(vals):
        if v > 0.12:
            ax.text(left[i] + v/2, i, f"{label.split()[0]}\n{v:.2f}", ha="center", va="center", fontsize=10)
    left += vals

ax.set_title("Replica placement policy across heterogeneous nodes")
ax.set_xlabel("placement fraction")
ax.set_xlim(0, 1)
ax.grid(True, axis="x", alpha=0.25)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.24), ncol=4)

savefig("59_replica_placement_policy_v3.png")

## 9. Capstone: Notebook 59 closes the second systems arc

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Notebook 59 closes the second systems arc", pad=22)

first = [
    ("00\nContext", 0.04),
    ("07\nVerification\nResource", 0.18),
    ("13\nConfidence\nScheduling", 0.32),
    ("17\nSemi-AR\nDrafting", 0.46),
    ("23\nThroughput\nObjective", 0.60),
    ("29\nHardware\nConstraints", 0.74),
    ("37\nOperating\nRegimes", 0.88),
]
for label, x0 in first:
    rounded_box(ax, (x0, 0.67), 0.12, 0.10, label, fontsize=10)
for (_, x1), (_, x2) in zip(first[:-1], first[1:]):
    arrow(ax, (x1+0.12, 0.72), (x2, 0.72), lw=1.5)

ax.text(0.50, 0.61, "first arc: mechanism → operating regime", ha="center", fontsize=15)
ax.plot([0.08, 0.92], [0.56, 0.56], color="black", linewidth=1.5)

second = [
    ("43\nResource\nAllocation", 0.28),
    ("49\nAdaptive\nInfrastructure", 0.43),
    ("53\nDistributed\nScheduling", 0.58),
    ("59\nCluster\nOptimization", 0.73),
]
for label, x0 in second:
    rounded_box(ax, (x0, 0.36), 0.16, 0.11, label, fontsize=11)
for (_, x1), (_, x2) in zip(second[:-1], second[1:]):
    arrow(ax, (x1+0.16, 0.415), (x2, 0.415), lw=1.5)

ax.text(
    0.50, 0.27,
    "second arc: controller → infrastructure → distributed scheduling → cluster optimization",
    ha="center", fontsize=15
)

rounded_box(ax, (0.40, 0.08), 0.20, 0.09, "next arc:\nlearning controllers", fontsize=14)
arrow(ax, (0.81, 0.36), (0.54, 0.17), "arc3,rad=-0.25")

savefig("59_repository_synthesis_v3.png")

## 10. Final synthesis

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Cluster optimization specifies global adaptive serving", pad=22)

items = [
    "distributed\nschedulers",
    "cluster\ntelemetry",
    "cluster\nstate S(t)",
    "objective",
    "global\noptimizer",
    "policy",
    "adaptive\nserving",
]
xs = np.linspace(0.05, 0.83, len(items))
for x0, label in zip(xs, items):
    rounded_box(ax, (x0, 0.53), 0.13, 0.13, label, fontsize=13)
for x1, x2 in zip(xs[:-1], xs[1:]):
    arrow(ax, (x1+0.13, 0.595), (x2, 0.595), lw=1.6)

rounded_box(
    ax, (0.21, 0.20), 0.58, 0.11,
    "Notebook 59 closes the second systems arc:\nlocal adaptive serving becomes cluster-level optimization.",
    fontsize=15
)

savefig("59_final_synthesis_v3.png")

## Download generated figures

In [ ]:
zip_path = "notebook_59_cluster_optimization_v3_figures.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for filename in sorted(os.listdir(FIG_DIR)):
        if filename.startswith("59_") and filename.endswith(".png"):
            zf.write(os.path.join(FIG_DIR, filename), arcname=filename)

print(f"Created {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download helper is available in Colab. Outside Colab, download the ZIP from the working directory.")

## Summary

Notebook 59 completes the second systems arc:

\[
\text{controller}
\rightarrow
\text{infrastructure}
\rightarrow
\text{distributed scheduling}
\rightarrow
\text{cluster optimization}
\]

The next arc can now move from designed controller policies to learned controller policies.